# 03 — Optuna: búsqueda bayesiana de hiperparámetros

**Entrada** : `{BUCKET}/datasets_fe/<dataset_fe>.parquet` (salida de `02_FE`)
**Salida**  : `{BUCKET}/exp/<EXPERIMENTO>/` — hiperparámetros, trials, importancia de variables, gráficos
**Leaderboard**: `{BUCKET}/exp/leaderboard.csv` — una fila por experimento, para comparar entre sí

Cómo funciona:

1. **El nombre del experimento sale del nombre del dataset + la variable respuesta elegida.**
   Todas las palancas de `01` y `02` viven en el nombre del parquet; este notebook lo parsea
   y le suma la palanca propia (qué columna `clase_*` se predice). Así cada combinación
   tiene su carpeta, su study de Optuna y su fila en el leaderboard, sin pisarse.
2. **Los meses de train / validation / test se declaran a mano** en `PARAM`. Optuna optimiza
   contra `val`; `test` es holdout y se mide una sola vez al final. Cambiar la partición
   genera otro experimento (los meses entran en el nombre), así que no se pisan entre sí.
3. **Control de data leakage** antes de entrenar (celda 4). Si algo falla, el notebook corta.
4. **WAPE siempre en toneladas**, reconstruyendo el nivel si la respuesta es `_norm` o `_delta`,
   y agregando por producto (que es como se evalúa la competencia).
5. **Las predicciones salen con `product_id` / `customer_id` / `Agrupacion_ID`** y el mes
   objetivo, para poder armar la entrega.

## 0 — Ambiente

In [ ]:
!pip install -q uv
!uv pip install -q pyarrow polars lightgbm optuna sqlalchemy plotly matplotlib

In [ ]:
import json, os, re, shutil, sys
from pathlib import Path

import numpy as np
import polars as pl


# ── RUTAS DEL BUCKET ──────────────────────────────────────────────────────
# Colab -> /content/buckets/b1     |     VM -> /home/ds/buckets/b1
def resolver_bucket() -> Path:
    # 1) LABO3_BUCKET: para correr fuera de la nube (server propio, notebook local).
    #    export LABO3_BUCKET=/ruta/a/mi/bucket   (o os.environ[...] antes de esta celda)
    env = os.environ.get("LABO3_BUCKET")
    if env:
        p = Path(env).expanduser().resolve()
        p.mkdir(parents=True, exist_ok=True)
        return p
    # 2) rutas conocidas: Colab y la VM de GCP
    for cand in ("/content/buckets/b1", "/home/ds/buckets/b1"):
        if Path(cand).is_dir():
            return Path(cand)
    raise RuntimeError(
        "No encontre el bucket. Opciones: "
        "(a) Colab / VM de GCP -> corre la celda de init del ambiente; "
        "(b) server propio o local -> defini LABO3_BUCKET antes de esta celda, ej. "
        "os.environ['LABO3_BUCKET'] = '/home/usuario/labo3-bucket'"
    )


BUCKET  = resolver_bucket()
RUTA_FE = BUCKET / "datasets_fe"          # entrada: lo que dejo 02_FE
RUTA_EXP = BUCKET / "exp"                 # salida: una carpeta por experimento
RUTA_EXP.mkdir(parents=True, exist_ok=True)

print(f"BUCKET : {BUCKET}")
print(f"\nDatasets FE disponibles en {RUTA_FE}:")
for p in sorted(RUTA_FE.glob("*.parquet")):
    print(f"  - {p.name}")

## 1 — Palancas

Lo único que se toca a mano: **qué dataset FE** y **qué variable respuesta**. El resto del
nombre del experimento se deduce solo.

Las tres respuestas posibles que deja `02_FE` (elegir UNA, las otras se excluyen de las features):

| `TARGET` | qué es | reconstrucción a toneladas |
|---|---|---|
| `clase_tn`       | toneladas a `t+2`, escala original | ninguna |
| `clase_tn_norm`  | la misma, normalizada con `B0`/`B1` de la ventana de lags | desnormalizar (`lag=-2`) |
| `clase_tn_delta` | `clase_tn_norm - tn0_norm` (cambio a 2 períodos) | sumar `tn0_norm` y desnormalizar |

El WAPE se mide **siempre en toneladas**, así los tres son comparables en el leaderboard.

In [ ]:
def rango_meses(desde: int, hasta: int) -> list:
    """Lista de periodos AAAAMM consecutivos, inclusive: rango_meses(201701, 201703)
    -> [201701, 201702, 201703]. Solo para no escribir 36 numeros a mano."""
    a, b = (desde // 100) * 12 + desde % 100, (hasta // 100) * 12 + hasta % 100
    return [((m - 1) // 12) * 100 + ((m - 1) % 12) + 1 for m in range(a, b + 1)]


PARAM = {
    # ── PALANCA 0: dataset de entrada (nombre exacto, de la lista de arriba) ──
    'dataset_fe': 'preprocesado_grpClienteProducto_fill0_denseLife_24lags_recta_2deltas.parquet',

    # ── PALANCA 1: VARIABLE RESPUESTA ────────────────────────────────────
    # 'clase_tn' | 'clase_tn_norm' | 'clase_tn_delta'
    'target': 'clase_tn_norm',

    # ── PALANCA 2: horizonte de prediccion (DEBE coincidir con 02_FE) ────
    # Define el GAP obligatorio entre train y validacion (control de leakage).
    'horizonte': 2,

    # ── PALANCA 3: MESES DE TRAIN / VALIDATION / TEST (hardcodeados) ─────
    # Los tres conjuntos se declaran a mano, con periodos AAAAMM explicitos.
    # Cada uno puede mover estas listas y comparar experimentos en el leaderboard.
    #
    #   train : lo que ve el modelo en cada trial de Optuna.
    #   val   : lo que Optuna MINIMIZA. Es la unica senial que guia la busqueda.
    #   test  : holdout. NO participa de Optuna; se mide UNA vez al final,
    #           con los hiperparametros ya elegidos. Es la estimacion honesta.
    #
    # REGLA DE ORO (la valida la celda de leakage, no se puede saltear):
    #   max(train) + horizonte <= min(val)      y      max(val) + horizonte <= min(test)
    # Una fila de mes t tiene como target tn(t+horizonte). Sin ese gap, el target de
    # las ultimas filas de train ES el periodo que se esta validando.
    #
    # Con horizonte=2 y datos hasta 201912 (supervisados hasta 201910):
    'meses_train': rango_meses(201701, 201905),
    'meses_val':   [201907, 201908],
    'meses_test':  [201910],

    # Reentrenar con train+val antes de medir test (mas datos, mismo hiperparametro).
    # False = medir test con el modelo entrenado solo en train.
    'reentrenar_con_val_para_test': True,

    # ── PALANCA 5: WAPE agregado por producto (como evalua la competencia) ──
    # True  -> se suman las predicciones de todos los clientes por producto y recien
    #          ahi se calcula el WAPE. Es la metrica del negocio.
    # False -> WAPE fila a fila (mas ruidosa, util para debug).
    'wape_por_producto': True,

    # ── PALANCA 6: objetivo de LightGBM ──────────────────────────────────
    # 'regression' | 'regression_l1' | 'tweedie' | 'poisson'
    # OJO: tweedie/poisson exigen target >= 0 -> solo con target='clase_tn'.
    'objective_lgbm': 'regression',
    'tweedie_optimizar': True,

    # ── PALANCA 7: trials nuevos por corrida (se acumulan en el study) ───
    'n_trials': 50,

    # ── PALANCA 8: regularizacion ('normal' | 'fuerte') ─────────────────
    'regularizacion': 'normal',

    # ── PALANCA 9: peso por recencia. None = todos los meses pesan igual ──
    'decay_recencia': None,

    # ── PALANCA 10: sampling de filas de train (None = todas) ───────────
    'sampling_frac': None,

    # ── PALANCA 11: features a EXCLUIR a mano (ademas de las prohibidas) ──
    'features_excluir': [],

    # ── PALANCA 12: sufijo libre para diferenciar corridas de la misma config ──
    'sufijo': '',

    'cols_categoricas': ['cat1', 'cat2', 'cat3', 'brand'],
    'semilla': 102191,
}

## 2 — Nombre del experimento (parseado del nombre del dataset)

El parquet de `02_FE` se llama:

```
preprocesado_{Agrupamiento}_{Completado}_{Formato}{Productos}_{MAX_LAGS}lags_{METODO}_{SALTO}deltas.parquet
```

De ahí salen `metodo` (necesario para desnormalizar) y `granularidad` (necesaria para saber si
hay `customer_id`), y el nombre del experimento. Si el nombre no matchea, el notebook corta:
sin `metodo` no se puede reconstruir el nivel en toneladas y el WAPE sería basura.

In [ ]:
PATRON_FE = re.compile(
    r"^preprocesado"
    r"_(?P<agrupamiento>grpClienteProducto|grpProducto)"
    r"_(?P<completado>fill0|fillNA)"
    r"_(?P<formato>denseLife|denseFull)"
    r"(?P<productos>_tgtFilter)?"
    r"(?P<muestra>_smpl\d+)?"
    r"_(?P<max_lags>\d+)lags"
    r"_(?P<metodo>recta|zscore|minmax|media)"
    r"_(?P<salto>\d+)deltas"
    r"\.parquet$"
)

TARGETS_VALIDOS = {'clase_tn': 'nivel', 'clase_tn_norm': 'norm', 'clase_tn_delta': 'delta'}

if PARAM['target'] not in TARGETS_VALIDOS:
    raise ValueError(f"target invalido: {PARAM['target']!r}. Opciones: {list(TARGETS_VALIDOS)}")

m = PATRON_FE.match(PARAM['dataset_fe'])
if m is None:
    raise ValueError(
        f"El nombre {PARAM['dataset_fe']!r} no sigue la convencion de 02_FE.\n"
        f"Esperado: preprocesado_{{grp}}_{{fill}}_{{dense}}[_tgtFilter][_smplN]_{{N}}lags_{{metodo}}_{{S}}deltas.parquet"
    )

CFG = m.groupdict()
CFG['max_lags'] = int(CFG['max_lags'])
CFG['salto']    = int(CFG['salto'])
CFG['productos'] = CFG['productos'] or ''
CFG['muestra'] = CFG['muestra'] or ''
# La granularidad manda: con 'pc' cada fila es producto-cliente; con 'p' ya viene agregado.
CFG['granularidad'] = 'pc' if CFG['agrupamiento'] == 'grpClienteProducto' else 'p'

METODO      = CFG['metodo']          # para desnormalizar
TARGET      = PARAM['target']
TARGET_KIND = TARGETS_VALIDOS[TARGET]  # 'nivel' | 'norm' | 'delta'

# ── Nombre del experimento: dataset + variable respuesta ────────────────
# Los meses de val/test entran en el nombre: cambiar la particion es OTRO experimento
# y no debe pisar la carpeta ni el study del anterior.
_v = PARAM['meses_val']
_t = PARAM['meses_test']
TAG_SPLIT = f"val{_v[0]}-{_v[-1]}_test{_t[0]}-{_t[-1]}"

# Las features excluidas TAMBIEN definen el experimento: sacar B0/B1 es otro modelo,
# no la misma corrida. Sin esta marca, dos configs distintas compartirian carpeta y
# study de Optuna, y la segunda pisaria a la primera sin avisar.
_ex = sorted(PARAM['features_excluir'])
TAG_EXCL = ("__excl-" + "-".join(_ex)) if 0 < len(_ex) <= 3 else (f"__excl{len(_ex)}feats" if _ex else "")

EXPERIMENTO = (
    f"{CFG['agrupamiento']}_{CFG['completado']}_{CFG['formato']}{CFG['productos']}{CFG['muestra']}"
    f"_{CFG['max_lags']}lags_{CFG['metodo']}_{CFG['salto']}deltas"
    f"__y-{TARGET_KIND}"
    f"__{PARAM['objective_lgbm']}"
    f"__{TAG_SPLIT}"
    + TAG_EXCL
    + (f"__{PARAM['sufijo']}" if PARAM['sufijo'] else "")
)

DIR_OUT = RUTA_EXP / EXPERIMENTO
DIR_OUT.mkdir(parents=True, exist_ok=True)

# El .db de Optuna va LOCAL (sqlite sobre el drive montado se traba) y se respalda al bucket.
DB_LOCAL  = Path.home() / f"optuna_{EXPERIMENTO}.db"
DB_BUCKET = RUTA_EXP / "optuna_db" / f"{EXPERIMENTO}.db"
DB_BUCKET.parent.mkdir(parents=True, exist_ok=True)
# Si ya hay un backup en el bucket y no hay db local, la recupero (VM recreada).
if DB_BUCKET.exists() and not DB_LOCAL.exists():
    shutil.copy(DB_BUCKET, DB_LOCAL)
    print(f"Study recuperado del bucket: {DB_BUCKET}")

STORAGE = f"sqlite:///{DB_LOCAL}"

print(f"Config del dataset : {CFG}")
print(f"Variable respuesta : {TARGET}  (kind={TARGET_KIND})")
print(f"\nEXPERIMENTO        : {EXPERIMENTO}")
print(f"Carpeta de salida  : {DIR_OUT}")
print(f"Study storage      : {STORAGE}")

## 3 — Carga del dataset

In [ ]:
path_in = RUTA_FE / PARAM['dataset_fe']
if not path_in.exists():
    raise FileNotFoundError(f"No existe {path_in}. Corre 02_FE con esas palancas.")

df = pl.read_parquet(path_in)
print(f"Dataset: {df.shape[0]:,} filas x {df.shape[1]} columnas")

if TARGET not in df.columns:
    raise ValueError(
        f"El dataset no tiene la columna {TARGET!r}. Columnas clase_* disponibles: "
        f"{[c for c in df.columns if c.startswith('clase_')]}"
    )

periodos = sorted(df['periodo'].unique().to_list())
print(f"Periodos: {periodos[0]} -> {periodos[-1]}  ({len(periodos)} meses)")

# Filas con target nulo = los ultimos `horizonte` periodos (todavia no se conoce t+2).
# Son las filas de INFERENCIA: se apartan aca y no entran ni a train ni a validacion.
mask_infer = pl.col(TARGET).is_null()
n_infer = df.filter(mask_infer).height
df_infer = df.filter(mask_infer)
df_sup   = df.filter(~mask_infer)

print(f"\nFilas supervisadas (target no nulo): {df_sup.height:,}")
print(f"Filas de inferencia (target nulo)  : {n_infer:,}"
      f"  -> periodos {sorted(df_infer['periodo'].unique().to_list())}")

## 4 — Control de data leakage

Los meses de train / val / test están **hardcodeados en `PARAM`**, así que este notebook no
inventa la partición: la **verifica**. Si algo no cierra, corta antes de entrenar.

1. **Ninguna `clase_*` es feature** — ni las otras dos, ni el propio target (es la `y`, no una `X`).
   Los `B0`/`B1` sí son features legítimas: se calculan solo con la ventana de lags pasados.
2. **Gap temporal entre conjuntos** — con `horizonte=2`, una fila de período `t` tiene como
   target `tn(t+2)`. Si train llega hasta `min(val) - 1`, el target de esas filas **es** el
   período que se está validando. Por eso se exige
   `max(train) + horizonte <= min(val)` y `max(val) + horizonte <= min(test)`.
   Este es el bug clásico del pipe anterior, que cortaba en `periodos[-2]`.
3. **Los tres conjuntos son disjuntos** y están en orden cronológico `train < val < test`.
4. **Correlación casi perfecta** feature↔target (`|r| > 0.999`) — una feature que *es* el
   target disfrazado.
5. **Filas de inferencia apartadas** — las de `clase_*` nula (los últimos `horizonte` meses)
   no entran a train, val ni test: son las que se van a predecir.
6. **Sin nulos del target** dentro de los conjuntos supervisados.

**Test nunca entra a Optuna.** Optuna minimiza el WAPE de `val`; `test` se mide una sola vez
al final. Si la brecha `test - val` es grande, hubo sobreajuste a la validación.

In [ ]:
# ── 4.1 Features: que entra y que no ─────────────────────────────────────
# Prohibido: identificadores, el eje temporal, y TODAS las columnas clase_*.
# Ojo: el propio TARGET tambien queda fuera de FEATURES (es la y, no una X); y las otras
# dos clase_* son el mismo dato en otra escala -> usarlas seria leakage puro.
COLS_ID = ['product_id', 'customer_id', 'Agrupacion_ID', 'periodo']
COLS_CLASE = [c for c in df.columns if c.startswith('clase_')]
COLS_PROHIBIDAS = set(COLS_ID) | set(COLS_CLASE)

FEATURES = [
    c for c in df.columns
    if c not in COLS_PROHIBIDAS and c not in PARAM['features_excluir']
]
# Categoricas: las declaradas + cualquier columna de texto que se haya colado
# (ej. un 'brand_right' de un join). LightGBM no come objects: o son categoria o revientan.
TIPOS_TEXTO = (pl.Utf8, pl.String, pl.Categorical, pl.Enum, pl.Boolean)
CAT_AUTO = [c for c in FEATURES
            if df.schema[c] in TIPOS_TEXTO and c not in PARAM['cols_categoricas']]
CAT_FEATURES = [c for c in PARAM['cols_categoricas'] if c in FEATURES] + CAT_AUTO

print(f"Features ({len(FEATURES)})")
print(f"Categoricas declaradas: {[c for c in PARAM['cols_categoricas'] if c in FEATURES]}")
if CAT_AUTO:
    print(f"Categoricas AUTODETECTADAS (columnas de texto no declaradas): {CAT_AUTO}")
print(f"\nExcluidas por prohibidas ({len(COLS_PROHIBIDAS)}): {sorted(COLS_PROHIBIDAS)}")
if PARAM['features_excluir']:
    print(f"Excluidas a mano: {PARAM['features_excluir']}")

In [ ]:
# ── 4.2 Utilidades de periodo (enteros AAAAMM) ───────────────────────────
def a_indice_mes(p: int) -> int:
    return (p // 100) * 12 + (p % 100) - 1


def desplazar_meses(p: int, k: int) -> int:
    m = a_indice_mes(p) + k
    return (m // 12) * 100 + (m % 12) + 1


assert desplazar_meses(201912, -2) == 201910
assert desplazar_meses(201901, -1) == 201812


# ── 4.3 Los tres conjuntos, tal como se declararon en PARAM ──────────────
H = PARAM['horizonte']
periodos_sup = sorted(df_sup['periodo'].unique().to_list())
set_sup = set(periodos_sup)

# Se recortan a lo que existe en el dataset supervisado (avisa si algo se cae).
MESES_TRAIN = sorted(set(PARAM['meses_train']) & set_sup)
MESES_VAL   = sorted(set(PARAM['meses_val'])   & set_sup)
MESES_TEST  = sorted(set(PARAM['meses_test'])  & set_sup)

faltan = {
    'train': sorted(set(PARAM['meses_train']) - set_sup),
    'val':   sorted(set(PARAM['meses_val'])   - set_sup),
    'test':  sorted(set(PARAM['meses_test'])  - set_sup),
}

for nombre, ms in (('TRAIN', MESES_TRAIN), ('VAL', MESES_VAL), ('TEST', MESES_TEST)):
    if not ms:
        raise ValueError(
            f"{nombre} quedo vacio. Periodos supervisados disponibles: "
            f"{periodos_sup[0]}..{periodos_sup[-1]}. Revisa PARAM['meses_{nombre.lower()}']."
        )
    print(f"{nombre:6s} ({len(ms):2d} meses): {ms[0]} .. {ms[-1]}   {ms if len(ms) <= 6 else ''}")

for k, v in faltan.items():
    if v:
        print(f"\n  aviso: meses de {k} que NO estan en el dataset y se ignoran: {v}")

sin_asignar = sorted(set_sup - set(MESES_TRAIN) - set(MESES_VAL) - set(MESES_TEST))
if sin_asignar:
    print(f"\n  meses supervisados sin asignar a ningun conjunto (quedan sin usar): {sin_asignar}")

In [ ]:
# ── 4.4 Los chequeos ─────────────────────────────────────────────────────
leak = {'errores': [], 'warnings': [], 'ok': []}


def _err(msg):
    leak['errores'].append(msg)
    print(f"  [ERROR]   {msg}")


def _warn(msg):
    leak['warnings'].append(msg)
    print(f"  [WARNING] {msg}")


def _ok(msg):
    leak['ok'].append(msg)
    print(f"  [ok]      {msg}")


print("CONTROL DE DATA LEAKAGE")
print("=" * 72)

# 1) ninguna columna prohibida entre las features
intrusas = sorted(set(FEATURES) & COLS_PROHIBIDAS)
if intrusas:
    _err(f"columnas prohibidas dentro de FEATURES: {intrusas}")
else:
    _ok(f"ninguna de las {len(COLS_PROHIBIDAS)} columnas prohibidas esta en FEATURES")

clase_en_x = sorted(set(FEATURES) & set(COLS_CLASE))
if clase_en_x:
    _err(f"columnas clase_* usadas como feature: {clase_en_x}")
else:
    _ok(f"ninguna clase_* es feature; {TARGET} se usa solo como target "
        f"(las otras {len(COLS_CLASE)-1} quedaron afuera)")

# 2) gap temporal entre conjuntos consecutivos
for a, b, na, nb in ((MESES_TRAIN, MESES_VAL, 'train', 'val'),
                     (MESES_VAL, MESES_TEST, 'val', 'test')):
    gap = a_indice_mes(min(b)) - a_indice_mes(max(a))
    if gap < H:
        _err(f"gap {na}->{nb} = {gap} mes(es) < horizonte {H}: el target de las filas de "
             f"{na} en {max(a)} es tn({desplazar_meses(max(a), H)}), que cae dentro de {nb} "
             f"(empieza en {min(b)}). Atrasa max({na}) a {desplazar_meses(min(b), -H)} o antes.")
    else:
        _ok(f"gap {na}({max(a)}) -> {nb}({min(b)}) = {gap} mes(es) >= horizonte {H}")

# 2b) train+val no pueden alcanzar test cuando se reentrena para el holdout
if PARAM['reentrenar_con_val_para_test']:
    gap_tv = a_indice_mes(min(MESES_TEST)) - a_indice_mes(max(MESES_TRAIN + MESES_VAL))
    if gap_tv < H:
        _err(f"reentrenar_con_val_para_test=True pero el gap (train+val)->test es {gap_tv} < {H}")
    else:
        _ok(f"gap (train+val)({max(MESES_TRAIN+MESES_VAL)}) -> test({min(MESES_TEST)}) "
            f"= {gap_tv} >= {H}")

# 3) los tres conjuntos son disjuntos
for (na, a), (nb, b) in ((('train', MESES_TRAIN), ('val', MESES_VAL)),
                         (('train', MESES_TRAIN), ('test', MESES_TEST)),
                         (('val', MESES_VAL),     ('test', MESES_TEST))):
    inter = sorted(set(a) & set(b))
    if inter:
        _err(f"{na} y {nb} comparten los meses {inter}")
    else:
        _ok(f"{na} y {nb} son disjuntos")

# 3b) orden cronologico: train < val < test (validar hacia atras es trampa)
if max(MESES_TRAIN) >= min(MESES_VAL):
    _err(f"train llega a {max(MESES_TRAIN)}, igual o posterior al inicio de val {min(MESES_VAL)}")
if max(MESES_VAL) >= min(MESES_TEST):
    _err(f"val llega a {max(MESES_VAL)}, igual o posterior al inicio de test {min(MESES_TEST)}")
if max(MESES_TRAIN) < min(MESES_VAL) and max(MESES_VAL) < min(MESES_TEST):
    _ok("orden cronologico correcto: train < val < test")

# 4) correlacion casi perfecta feature <-> target
y_chk = df_sup[TARGET].to_numpy().astype(np.float64)
sospechosas = []
num_feats = [c for c in FEATURES
             if df_sup.schema[c] in (pl.Float32, pl.Float64, pl.Int8, pl.Int16,
                                     pl.Int32, pl.Int64, pl.UInt8, pl.UInt16,
                                     pl.UInt32, pl.UInt64)]
for c in num_feats:
    x = df_sup[c].to_numpy().astype(np.float64)
    ok = np.isfinite(x) & np.isfinite(y_chk)
    if ok.sum() < 100:
        continue
    xs, ys = x[ok], y_chk[ok]
    if xs.std() == 0 or ys.std() == 0:
        continue
    r = float(np.corrcoef(xs, ys)[0, 1])
    if abs(r) > 0.999:
        sospechosas.append((c, round(r, 6)))

if sospechosas:
    _err(f"features con |corr| > 0.999 contra el target (son el target disfrazado): {sospechosas}")
else:
    _ok(f"ninguna de las {len(num_feats)} features numericas correlaciona >0.999 con el target")
leak['correlaciones_sospechosas'] = sospechosas

# 5) filas de inferencia apartadas
if df_sup.filter(pl.col(TARGET).is_null()).height:
    _err("quedaron filas con target nulo en el set supervisado")
else:
    _ok(f"{n_infer:,} filas de inferencia (target nulo) apartadas en df_infer")

# 6) periodos de inferencia no aparecen en train / val / test
periodos_infer = set(df_infer['periodo'].unique().to_list())
solapa = sorted(periodos_infer & (set(MESES_TRAIN) | set(MESES_VAL) | set(MESES_TEST)))
if solapa:
    _err(f"periodos de inferencia usados en train/val/test: {solapa}")
else:
    _ok("los periodos de inferencia no se usan para entrenar, validar ni testear")

leak['meses'] = {'train': MESES_TRAIN, 'val': MESES_VAL, 'test': MESES_TEST,
                 'inferencia': sorted(periodos_infer), 'sin_asignar': sin_asignar,
                 'horizonte': H}

print("=" * 72)
leak['resumen'] = f"{len(leak['errores'])} error(es), {len(leak['warnings'])} warning(s)"
print(leak['resumen'])

with open(DIR_OUT / 'leakage_report.json', 'w', encoding='utf-8') as f:
    json.dump(leak, f, indent=2, ensure_ascii=False)

if leak['errores']:
    raise RuntimeError(
        "Control de data leakage FALLIDO. Revisa los errores de arriba "
        f"(detalle en {DIR_OUT/'leakage_report.json'})."
    )
print(f"\nControl superado. Reporte: {DIR_OUT/'leakage_report.json'}")

## 5 — Métrica: WAPE en toneladas

Sea cual sea la variable respuesta, primero se **reconstruye el nivel en toneladas** y recién
ahí se mide. Es la única forma de que `y-nivel`, `y-norm` e `y-delta` sean comparables en el
leaderboard.

```
nivel  = pred                                   si target = clase_tn
nivel  = desnorm(pred, lag=-2)                  si target = clase_tn_norm
nivel  = desnorm(pred + tn0_norm, lag=-2)       si target = clase_tn_delta
```

`desnorm` es el inverso exacto de `normalizar_achatado` de `02_FE`, con los `B0`/`B1` que el
propio dataset trae por fila.

In [ ]:
def reconstruir_nivel(pred, df_ctx: pl.DataFrame) -> np.ndarray:
    """Pasa la prediccion del modelo a toneladas, segun la variable respuesta elegida.

    df_ctx debe traer B0, B1 y (si target='clase_tn_delta') tn0_norm, alineadas por fila.
    """
    pred = np.asarray(pred, dtype=np.float64)

    if TARGET_KIND == 'nivel':
        return pred

    if TARGET_KIND == 'delta':
        # clase_tn_delta = clase_tn_norm - tn0_norm  ->  volvemos a la escala normalizada
        pred = pred + df_ctx['tn0_norm'].to_numpy().astype(np.float64)

    B0 = df_ctx['B0'].to_numpy().astype(np.float64)
    B1 = df_ctx['B1'].to_numpy().astype(np.float64)

    if METODO == 'recta':
        # inverso de: norm = valor - (B0 + B1 * lag);  la clase esta en lag = -2
        return pred + (B0 + B1 * (-2.0))
    # zscore | minmax | media: inverso de norm = (valor - B0) / B1, con el mismo B1 seguro
    B1_safe = np.where((B1 == 0) | ~np.isfinite(B1), 1.0, B1)
    return pred * B1_safe + B0


def wape(y_real_tn, y_pred_tn, product_ids=None, por_producto=True) -> float:
    """WAPE en toneladas. Con por_producto=True agrega por product_id primero
    (asi lo evalua la competencia: el error es sobre el total vendido de cada producto)."""
    y_real = np.asarray(y_real_tn, dtype=np.float64)
    y_pred = np.maximum(np.asarray(y_pred_tn, dtype=np.float64), 0.0)  # no hay ventas negativas

    if por_producto and product_ids is not None:
        ids = np.asarray(product_ids)
        orden, inv = np.unique(ids, return_inverse=True)
        y_real = np.bincount(inv, weights=y_real, minlength=len(orden))
        y_pred = np.bincount(inv, weights=y_pred, minlength=len(orden))

    den = np.abs(y_real).sum()
    return float('nan') if den == 0 else float(np.abs(y_real - y_pred).sum() / den)


# Chequeo de sanidad: reconstruir el TARGET REAL tiene que devolver clase_tn.
_m = df_sup.head(20_000)
_rec = reconstruir_nivel(_m[TARGET].to_numpy(), _m)
_err_max = float(np.nanmax(np.abs(_rec - _m['clase_tn'].to_numpy())))
print(f"Round-trip de reconstruccion (target -> toneladas): error maximo = {_err_max:.10f}")
if _err_max > 1e-6:
    raise RuntimeError(
        f"La reconstruccion a toneladas no cierra (error {_err_max}). "
        f"Revisa que METODO={METODO!r} sea el que uso 02_FE."
    )
print("Reconstruccion validada: el WAPE se mide en toneladas reales.")

## 6 — Función objetivo y búsqueda bayesiana

In [ ]:
import lightgbm as lgb
import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)

if PARAM['objective_lgbm'] in ('tweedie', 'poisson') and TARGET_KIND != 'nivel':
    raise ValueError(
        f"objective_lgbm={PARAM['objective_lgbm']!r} exige target >= 0, "
        f"pero target={TARGET!r} tiene valores negativos. Usa target='clase_tn'."
    )

# ── Columnas de contexto: NO son features, pero viajan con cada fila ────────
#   - B0/B1/tn0_norm : necesarias para reconstruir toneladas
#   - clase_tn       : el nivel real, contra el que se mide el WAPE
#   - IDS            : product_id / customer_id / Agrupacion_ID -> sin esto no se sabe
#                      a que SKU corresponde cada prediccion. Se guardan en todos los
#                      parquet de salida (val, test e inferencia).
IDS = [c for c in ['product_id', 'customer_id', 'Agrupacion_ID'] if c in df.columns]
COLS_CTX = [c for c in ['B0', 'B1', 'tn0_norm', 'clase_tn'] + IDS if c in df.columns]

print(f"Identificadores que se arrastran a las predicciones: {IDS}")

# A pandas una sola vez (LightGBM + categoricas nativas)
_cols_pd = sorted(set(FEATURES + COLS_CTX + [TARGET, 'periodo']))
df_pd = df_sup.select(_cols_pd).to_pandas()
# Las filas de inferencia (target nulo) no tienen clase_tn; van por separado.
df_infer_pd = df_infer.select([c for c in _cols_pd if c in df_infer.columns]).to_pandas()
for c in CAT_FEATURES:
    df_pd[c] = df_pd[c].astype('category')
    if c in df_infer_pd.columns:
        df_infer_pd[c] = df_infer_pd[c].astype('category').cat.set_categories(
            df_pd[c].cat.categories)

print(f"df_pd (supervisado): {df_pd.shape}   |   df_infer_pd: {df_infer_pd.shape}")


def pesos_recencia(periodos_serie, decay):
    """Peso por recencia: el mes mas nuevo pesa 1, cada mes hacia atras decae x`decay`."""
    if decay is None:
        return None
    ps = sorted(periodos_serie.unique())
    idx = {p: i for i, p in enumerate(ps)}
    n = len(ps)
    return periodos_serie.map(lambda p: decay ** (n - 1 - idx[p])).values


def espacio_hiper(trial):
    base = {
        'objective':      PARAM['objective_lgbm'],
        'metric':         'mae',
        'verbosity':      -1,
        'boosting_type':  'gbdt',
        'seed':           PARAM['semilla'],
        'subsample_freq': 1,
        'n_jobs':         -1,
    }
    if PARAM['regularizacion'] == 'fuerte':
        base.update({
            'num_leaves':        trial.suggest_int('num_leaves', 8, 64),
            'max_depth':         trial.suggest_int('max_depth', 3, 7),
            'learning_rate':     trial.suggest_float('learning_rate', 5e-3, 0.1, log=True),
            'n_estimators':      trial.suggest_int('n_estimators', 100, 800),
            'min_child_samples': trial.suggest_int('min_child_samples', 30, 200),
            'subsample':         trial.suggest_float('subsample', 0.5, 0.9),
            'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.5, 0.9),
            'reg_alpha':         trial.suggest_float('reg_alpha', 0.1, 20.0, log=True),
            'reg_lambda':        trial.suggest_float('reg_lambda', 0.1, 20.0, log=True),
        })
    else:
        base.update({
            'num_leaves':        trial.suggest_int('num_leaves', 20, 300),
            'max_depth':         trial.suggest_int('max_depth', 3, 12),
            'learning_rate':     trial.suggest_float('learning_rate', 1e-3, 0.3, log=True),
            'n_estimators':      trial.suggest_int('n_estimators', 100, 2000),
            'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
            'subsample':         trial.suggest_float('subsample', 0.5, 1.0),
            'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.5, 1.0),
            'reg_alpha':         trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
            'reg_lambda':        trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        })
    if PARAM['objective_lgbm'] == 'tweedie' and PARAM['tweedie_optimizar']:
        base['tweedie_variance_power'] = trial.suggest_float('tweedie_variance_power', 1.1, 1.9)
    return base


def entrenar(params, meses_tr):
    """Entrena con las filas cuyo periodo esta en `meses_tr`. Devuelve el modelo."""
    df_tr = df_pd[df_pd['periodo'].isin(meses_tr)]
    if len(df_tr) == 0:
        raise ValueError(f'Sin filas de entrenamiento para los meses {meses_tr}')
    if PARAM['sampling_frac'] is not None:
        df_tr = df_tr.sample(frac=PARAM['sampling_frac'], random_state=PARAM['semilla'])

    modelo = lgb.LGBMRegressor(**params)
    modelo.fit(df_tr[FEATURES], df_tr[TARGET],
               sample_weight=pesos_recencia(df_tr['periodo'], PARAM['decay_recencia']),
               categorical_feature=CAT_FEATURES)
    return modelo


def predecir(modelo, df_eval):
    """Devuelve el DataFrame de evaluacion con ids, prediccion cruda y toneladas."""
    pred = modelo.predict(df_eval[FEATURES])
    ctx = pl.from_pandas(df_eval[[c for c in COLS_CTX if c in df_eval.columns]]
                         .reset_index(drop=True))
    pred_tn = np.maximum(reconstruir_nivel(pred, ctx), 0.0)  # no hay ventas negativas

    out = df_eval[IDS + ['periodo']].copy()
    # A que mes corresponde la prediccion: la fila es de t, se predice t+horizonte.
    out['periodo_objetivo'] = out['periodo'].map(lambda p: desplazar_meses(p, H))
    out['y_pred_target'] = pred
    out['tn_pred'] = pred_tn
    if 'clase_tn' in df_eval.columns:
        out['tn_real'] = df_eval['clase_tn'].values
        out['y_real_target'] = df_eval[TARGET].values
    return out


def evaluar(modelo, meses_ev, por_mes=True):
    """WAPE en toneladas sobre `meses_ev`. Devuelve (wape_global, {mes: wape}, df_pred)."""
    df_ev = df_pd[df_pd['periodo'].isin(meses_ev)]
    if len(df_ev) == 0:
        return float('nan'), {}, None
    pred = predecir(modelo, df_ev)

    por_mes_d = {}
    if por_mes:
        for m in sorted(meses_ev):
            sub = pred[pred['periodo'] == m]
            if len(sub):
                por_mes_d[int(m)] = wape(sub['tn_real'], sub['tn_pred'],
                                         sub['product_id'], PARAM['wape_por_producto'])
    # WAPE global: todos los meses de evaluacion juntos (asi lo mide la competencia).
    glob = wape(pred['tn_real'], pred['tn_pred'], pred['product_id'],
                PARAM['wape_por_producto'])
    return glob, por_mes_d, pred


def objective(trial):
    """Optuna minimiza el WAPE en TONELADAS sobre MESES_VAL. Test no se toca aca."""
    modelo = entrenar(espacio_hiper(trial), MESES_TRAIN)
    score, por_mes, _ = evaluar(modelo, MESES_VAL)
    if np.isnan(score):
        raise optuna.TrialPruned()
    for m, v in por_mes.items():
        trial.set_user_attr(f'wape_val_{m}', v)
    return float(score)

In [ ]:
from tqdm.auto import tqdm

study = optuna.create_study(
    direction='minimize',
    sampler=optuna.samplers.TPESampler(seed=PARAM['semilla']),
    study_name=EXPERIMENTO,
    storage=STORAGE,
    load_if_exists=True,
)

print(f"Experimento    : {EXPERIMENTO}")
print(f"Trials previos : {len(study.trials)}")
print(f"Corriendo {PARAM['n_trials']} trials nuevos...\n")

with tqdm(total=PARAM['n_trials'], desc='Optuna') as pbar:
    def cb(st, tr):
        pbar.update(1)
        try:
            pbar.set_postfix({'mejor WAPE': f'{st.best_value:.5f}'})
        except ValueError:
            pass
    study.optimize(objective, n_trials=PARAM['n_trials'], callbacks=[cb])

print(f"\nTrials totales    : {len(study.trials)}")
print(f"Mejor WAPE (tn)   : {study.best_value:.5f}")
print(f"Mejores hiperparametros:")
for k, v in study.best_params.items():
    print(f"   {k:22s} {v}")

## 7 — Resultados del experimento

Dos modelos con los mismos hiperparámetros ganadores:

- **modelo de validación** — entrenado solo con `MESES_TRAIN`, medido en `MESES_VAL`.
- **modelo final** — entrenado con `train (+ val)`, medido **una vez** en `MESES_TEST`,
  y es el que produce la predicción a futuro.

Todo va a `{BUCKET}/exp/{EXPERIMENTO}/`:

| archivo | qué es |
|---|---|
| `resultado.json`                 | palancas, meses de cada conjunto, hiperparámetros, WAPE de val y test |
| `trials.csv`                     | todos los trials del study |
| `importancia.csv` / `.png`       | importancia de variables (gain y split) |
| `predicciones_val.parquet`       | ids + real vs predicho en toneladas, en validación |
| `predicciones_test.parquet`      | ídem en el holdout |
| `predicciones_inferencia.parquet`| ids + predicción de los meses sin target (lo que se entrega) |
| `prediccion_por_producto.csv`    | toneladas por `product_id` y mes objetivo |
| `leakage_report.json`            | el control de la celda 4, con los meses de cada conjunto |
| `optuna_*.html`                  | gráficos interactivos |

In [ ]:
import pandas as pd

mejores_params = espacio_hiper(optuna.trial.FixedTrial(study.best_params))

# ── Modelo de validacion: entrenado SOLO con train ───────────────────────
modelo_val = entrenar(mejores_params, MESES_TRAIN)
wape_val, wape_val_mes, pred_val = evaluar(modelo_val, MESES_VAL)

print(f"VALIDATION  meses {MESES_VAL}")
print(f"  WAPE global : {wape_val:.5f}")
for m, v in wape_val_mes.items():
    print(f"    {m}: {v:.5f}")

# ── Modelo de test: holdout, se mide UNA sola vez ────────────────────────
meses_fit_test = (MESES_TRAIN + MESES_VAL) if PARAM['reentrenar_con_val_para_test'] else MESES_TRAIN
modelo_final = entrenar(mejores_params, meses_fit_test)
wape_test, wape_test_mes, pred_test = evaluar(modelo_final, MESES_TEST)

print(f"\nTEST (holdout)  meses {MESES_TEST}   [entrenado con {len(meses_fit_test)} meses]")
print(f"  WAPE global : {wape_test:.5f}")
for m, v in wape_test_mes.items():
    print(f"    {m}: {v:.5f}")

_brecha = wape_test - wape_val
print(f"\nBrecha test - val: {_brecha:+.5f}"
      + ("   <- ojo, el test empeora bastante: posible sobreajuste a val"
         if _brecha > 0.02 else ""))

# ── Baseline naive: repetir tn0 (lo vendido en t). Sobre los MISMOS meses ──
def wape_naive_en(meses):
    _n = df_pd[df_pd['periodo'].isin(meses)]
    if len(_n) == 0 or 'tn0' not in _n.columns:
        return float('nan')
    return wape(_n['clase_tn'].to_numpy(), _n['tn0'].to_numpy(),
                _n['product_id'].to_numpy(), PARAM['wape_por_producto'])


naive_val, naive_test = wape_naive_en(MESES_VAL), wape_naive_en(MESES_TEST)
print(f"\nBaseline naive (repetir tn0):  val {naive_val:.5f}   test {naive_test:.5f}")
print(f"Mejora del modelo vs naive  :  val {100*(naive_val-wape_val)/naive_val:+.1f}%"
      f"   test {100*(naive_test-wape_test)/naive_test:+.1f}%")

# ── Predicciones con identificadores ─────────────────────────────────────
pl.from_pandas(pred_val).write_parquet(DIR_OUT / 'predicciones_val.parquet')
pl.from_pandas(pred_test).write_parquet(DIR_OUT / 'predicciones_test.parquet')
print(f"\nGuardadas predicciones val ({len(pred_val):,}) y test ({len(pred_test):,}) "
      f"con {IDS} + periodo + periodo_objetivo + tn_real + tn_pred")

### 7.1 — Predicción a futuro (con identificadores)

Las filas cuyo `clase_*` es nulo son los últimos `horizonte` meses: todavía no se conoce
`tn(t+2)`, así que **son las filas a predecir**. Se predicen con el modelo final y se guardan
con `product_id` / `customer_id` / `Agrupacion_ID` y el `periodo_objetivo` (= `t + horizonte`),
para poder armar la entrega después.

También se deja `prediccion_por_producto.csv`: las toneladas sumadas sobre todos los clientes
de cada producto para el mes objetivo — que es el formato en el que se entrega.

In [ ]:
if len(df_infer_pd) == 0:
    print("No hay filas de inferencia en este dataset.")
    pred_infer = None
else:
    pred_infer = predecir(modelo_final, df_infer_pd)

    print(f"Predicciones de inferencia: {len(pred_infer):,} filas")
    print(f"Periodo origen -> objetivo:")
    for p, po in sorted(set(zip(pred_infer['periodo'], pred_infer['periodo_objetivo']))):
        n = (pred_infer['periodo'] == p).sum()
        print(f"   {p} -> {po}   ({n:,} filas)")

    pl.from_pandas(pred_infer).write_parquet(DIR_OUT / 'predicciones_inferencia.parquet')

    # Entrega: toneladas por producto para cada mes objetivo (suma sobre clientes).
    entrega = (pred_infer.groupby(['periodo_objetivo', 'product_id'], as_index=False)['tn_pred']
               .sum().rename(columns={'tn_pred': 'tn'}).sort_values(['periodo_objetivo', 'product_id']))
    entrega.to_csv(DIR_OUT / 'prediccion_por_producto.csv', index=False)

    print(f"\nprediccion_por_producto.csv: {len(entrega):,} filas "
          f"({entrega['product_id'].nunique()} productos x "
          f"{entrega['periodo_objetivo'].nunique()} mes(es) objetivo)")
    print(entrega.head(10).to_string(index=False))

In [ ]:
# ── Importancia de variables ─────────────────────────────────────────────
imp = pd.DataFrame({
    'feature': modelo_final.feature_name_,
    'gain':    modelo_final.booster_.feature_importance(importance_type='gain'),
    'split':   modelo_final.booster_.feature_importance(importance_type='split'),
}).sort_values('gain', ascending=False).reset_index(drop=True)
imp['gain_pct'] = 100 * imp['gain'] / imp['gain'].sum()
imp['gain_pct_acum'] = imp['gain_pct'].cumsum()

imp.to_csv(DIR_OUT / 'importancia.csv', index=False)

n_utiles = int((imp['gain'] > 0).sum())
print(f"Features con gain > 0: {n_utiles} de {len(imp)}")
print(f"Top 10 concentra el {imp['gain_pct'].head(10).sum():.1f}% del gain\n")
print(imp.head(25).to_string(index=False))

sin_uso = imp[imp['gain'] == 0]['feature'].tolist()
if sin_uso:
    print(f"\n{len(sin_uso)} features que el modelo NUNCA uso (candidatas a features_excluir):")
    print(sin_uso[:40])

In [ ]:
import matplotlib.pyplot as plt

top = imp.head(30).iloc[::-1]
fig, ax = plt.subplots(figsize=(9, 9))
ax.barh(top['feature'], top['gain_pct'], color='#4C78A8')
ax.set_xlabel('% del gain total')
ax.set_title(f'Importancia de variables (top 30)\n{EXPERIMENTO}', fontsize=9)
ax.grid(axis='x', alpha=.3)
fig.tight_layout()
fig.savefig(DIR_OUT / 'importancia_top30.png', dpi=120)
plt.show()

In [ ]:
# ── Trials + graficos de Optuna ──────────────────────────────────────────
df_trials = study.trials_dataframe()
df_trials.to_csv(DIR_OUT / 'trials.csv', index=False)

print("Top 10 trials:")
print(df_trials[df_trials['state'] == 'COMPLETE']
      .sort_values('value')[['number', 'value']].head(10).to_string(index=False))

try:
    import optuna.visualization as vis
    for nombre, fig_ in [
        ('historia',           vis.plot_optimization_history(study)),
        ('importancia_hiper',  vis.plot_param_importances(study)),
        ('slice',              vis.plot_slice(study)),
    ]:
        fig_.write_html(str(DIR_OUT / f'optuna_{nombre}.html'))
        fig_.show()
except Exception as e:
    print(f"Graficos de Optuna no disponibles: {e}")

In [ ]:
# ── resultado.json: todo lo necesario para reproducir y para 04_entrenamiento_final ──
resultado = {
    'experimento':        EXPERIMENTO,
    'dataset_fe':         PARAM['dataset_fe'],
    'config_dataset':     CFG,
    'target':             TARGET,
    'target_kind':        TARGET_KIND,
    'metodo_normalizacion': METODO,
    'horizonte':          H,
    'meses_train':        MESES_TRAIN,
    'meses_val':          MESES_VAL,
    'meses_test':         MESES_TEST,
    'meses_inferencia':   sorted(periodos_infer),
    'reentrenar_con_val_para_test': PARAM['reentrenar_con_val_para_test'],
    'metrica':            'wape_toneladas' + ('_por_producto' if PARAM['wape_por_producto'] else '_por_fila'),
    'wape_mejor_trial':   study.best_value,
    'wape_val':           wape_val,
    'wape_val_por_mes':   wape_val_mes,
    'wape_test':          wape_test,
    'wape_test_por_mes':  wape_test_mes,
    'wape_naive_val':     naive_val,
    'wape_naive_test':    naive_test,
    'n_trials_total':     len(study.trials),
    'objective_lgbm':     PARAM['objective_lgbm'],
    'regularizacion':     PARAM['regularizacion'],
    'decay_recencia':     PARAM['decay_recencia'],
    'n_features':         len(FEATURES),
    'features':           FEATURES,
    'cat_features':       CAT_FEATURES,
    'features_excluir':   PARAM['features_excluir'],
    'features_sin_uso':   sin_uso,
    'hiperparametros':    study.best_params,
    'semilla':            PARAM['semilla'],
    'leakage':            leak['resumen'],
}

with open(DIR_OUT / 'resultado.json', 'w', encoding='utf-8') as f:
    json.dump(resultado, f, indent=2, ensure_ascii=False, default=str)

# Backup del study al bucket (el .db local se pierde si se destruye la VM)
shutil.copy(DB_LOCAL, DB_BUCKET)

print(f"Guardado en {DIR_OUT}:")
for p in sorted(DIR_OUT.iterdir()):
    print(f"  - {p.name}")
print(f"\nBackup del study: {DB_BUCKET}")

## 8 — Leaderboard: comparar experimentos entre sí

`{BUCKET}/exp/leaderboard.csv` acumula una fila por experimento (upsert por nombre), leyendo
los `resultado.json` de todas las carpetas. Es la vista para decidir qué combinación de
palancas conviene.

In [ ]:
filas = []
for d in sorted(RUTA_EXP.iterdir()):
    f_res = d / 'resultado.json'
    if not (d.is_dir() and f_res.exists()):
        continue
    r = json.load(open(f_res, encoding='utf-8'))
    _nt = r.get('wape_naive_test')
    filas.append({
        'experimento':   r['experimento'],
        'wape_test':     r.get('wape_test'),          # <- el numero honesto, ordena por este
        'wape_val':      r.get('wape_val'),
        'brecha_test_val': (round(r['wape_test'] - r['wape_val'], 5)
                            if r.get('wape_test') is not None and r.get('wape_val') is not None
                            else None),
        'wape_naive_test': _nt,
        'mejora_vs_naive_%': (round(100 * (_nt - r['wape_test']) / _nt, 2)
                              if _nt and r.get('wape_test') is not None else None),
        'meses_train':   f"{r['meses_train'][0]}-{r['meses_train'][-1]}" if r.get('meses_train') else None,
        'meses_val':     str(r.get('meses_val')),
        'meses_test':    str(r.get('meses_test')),
        'agrupamiento':  r['config_dataset']['agrupamiento'],
        'completado':    r['config_dataset']['completado'],
        'formato':       r['config_dataset']['formato'],
        'tgt_filter':    bool(r['config_dataset']['productos']),
        'max_lags':      r['config_dataset']['max_lags'],
        'norm':          r['config_dataset']['metodo'],
        'target':        r['target_kind'],
        'objective':     r['objective_lgbm'],
        'regularizacion': r['regularizacion'],
        'n_features':    r['n_features'],
        'n_trials':      r['n_trials_total'],
        'leakage':       r['leakage'],
    })

leaderboard = pd.DataFrame(filas).sort_values('wape_test').reset_index(drop=True)
leaderboard.to_csv(RUTA_EXP / 'leaderboard.csv', index=False)

print(f"{len(leaderboard)} experimento(s) en {RUTA_EXP/'leaderboard.csv'}\n")
leaderboard